# 🎯 1. Decorators — Fundamentals

Decorators are **the** most important Python-specific feature you'll encounter.
They let you modify or extend functions without changing their code.

| ☕ Java | 🐍 Python |
|---------|----------|
| `@Override` — metadata only, no behavior | `@decorator` — **modifies** the function |
| AOP / Proxy pattern for cross-cutting concerns | Decorators — built into the language |
| Spring `@Transactional`, `@Cacheable` | Custom decorators — you build them yourself |

**Key insight:** `@decorator` is syntactic sugar for `func = decorator(func)`

In [1]:
import time
from functools import wraps, lru_cache

## 1.1 Your First Decorator

A decorator is just a function that:
1. **Takes** a function as argument
2. **Returns** a new function (the wrapper)

The wrapper runs code before/after the original function.

In [ ]:
def simple_decorator(func):
    def wrapper(*args, **kwargs):
        print(" -> Before")
        result = func(*args, **kwargs)
        print(" -> After")
        return result
    return wrapper

In [6]:
# greed = simple_decorator(greed)
@simple_decorator
def greed(name: str) -> str:
    print(f"Hello, {name}!")
    return f"Greeted {name}"

In [4]:
result = greed("Alice")
print(result)

Hello, Alice!
Greeted Alice


In [ ]:
# new_greed = simple_decorator(greed)
# new_greed("Bob")

 -> Before
Hello, Bob!
 -> After


'Greeted Bob'

In [7]:
# with @decorator
result = greed("Alice")
print(result)

 -> Before
Hello, Alice!
 -> After
Greeted Alice


## 1.2 The Problem: Lost Metadata

Without `@wraps`, the decorated function **loses its identity**:

In [8]:
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

In [11]:
@bad_decorator
def my_function():
    """This is my important docstring ..."""
    pass

print(f"Name: {my_function.__name__}") # my_function
print(f"Docs: {my_function.__doc__}") # This is ... 
print("To not happen this look below @wraps")

Name: wrapper
Docs: None
To not happen this look below @wraps


## 1.3 `@wraps` — Always Use It

`functools.wraps` copies the original function's metadata to the wrapper.

**Rule:** Every decorator you write should use `@wraps(func)`.

In [18]:
def good_decorator(func):
    # @wrapper # This give error
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

In [19]:
@good_decorator
def my_function():
    """This is my important docstring ..."""
    pass

print(f"Name: {my_function.__name__}") 
print(f"Docs: {my_function.__doc__}")

Name: my_function
Docs: This is my important docstring ...


## 1.4 Practical: `@timer` Decorator

Measures how long a function takes — your first useful decorator.

In [22]:
def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

In [23]:
@timer
def slow_function():
    """Simulates a slow operation"""
    time.sleep(0.1)
    return "done"

result = slow_function()
print(f"Result: {result}")

slow_function took 0.1002s
Result: done


## 1.5 Practical: `@debug` Decorator

Logs function calls with arguments and return values.

In [24]:
def debug(func):
    """Log every call with arguments and return value."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        args_repr = [repr(a) for a in args]
        kwargs_repr = [f"{k}={v!r}" for k, v in kwargs.items()]
        signature = ", ".join(args_repr + kwargs_repr)
        print(f"  📞 {func.__name__}({signature})")
        result = func(*args, **kwargs)
        print(f"  📤 {func.__name__} → {result!r}")
        return result
    return wrapper
    
@debug
def add(a: int, b: int) -> int:
    return a + b

@debug
def greet(name: str, greeting: str = "Hello") -> str:
    return f"{greeting}, {name}!"

add(3, 5)

greet("Alice", greeting="Hi")

  📞 add(3, 5)
  📤 add → 8
  📞 greet('Alice', greeting='Hi')
  📤 greet → 'Hi, Alice!'


'Hi, Alice!'

## 1.6 Stacking Decorators

You can apply **multiple decorators** to one function.
They execute **bottom-up** (closest to the function first).

```python
@A       # 3rd: A wraps the result of B(C(func))
@B       # 2nd: B wraps the result of C(func)
@C       # 1st: C wraps func
def func(): ...

# Equivalent to: func = A(B(C(func)))
```

In [25]:
def bold(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<b>{func(*args, **kwargs)}</b>"
    return wrapper
 
def italic(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<i>{func(*args, **kwargs)}</i>"
    return wrapper
 
# Order matters!
@bold       # 2nd: wraps in <b>
@italic     # 1st: wraps in <i>
def say(text):
    return text
 
print(f"@bold @italic: {say('hello')}")
# → <b><i>hello</i></b>  (italic applied first, then bold)
 
# Swap order
@italic     # 2nd: wraps in <i>
@bold       # 1st: wraps in <b>
def say2(text):
    return text
 
print(f"@italic @bold: {say2('hello')}")
# → <i><b>hello</b></i>  (bold applied first, then italic)

@bold @italic: <b><i>hello</i></b>
@italic @bold: <i><b>hello</b></i>


## 1.7 Decorator Factory — Decorators with Arguments

When a decorator needs parameters, you add **one more layer** of nesting.
This is called a **decorator factory** — a function that *returns* a decorator.

```python
@retry(max_attempts=3)   # retry() returns the actual decorator
def func(): ...

# Equivalent to:
# decorator = retry(max_attempts=3)
# func = decorator(func)
```

In [26]:
import random
 
def retry(max_attempts: int = 3, delay: float = 0):
    """Decorator factory — retries a function on failure."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    print(f"  ⚠️ Attempt {attempt}/{max_attempts} failed: {e}")
                    if delay:
                        time.sleep(delay)
            raise last_exception
        return wrapper
    return decorator
 
@retry(max_attempts=5, delay=3)
def unreliable_api():
    if random.random() < 0.6:
        raise ConnectionError("Server timeout")
    return {"status": "ok"}
 
try:
    result = unreliable_api()
    print(f"  ✅ Success: {result}")
except ConnectionError:
    print("  ❌ Failed after all retries")

  ⚠️ Attempt 1/5 failed: Server timeout
  ⚠️ Attempt 2/5 failed: Server timeout
  ✅ Success: {'status': 'ok'}


### 📐 The 3 Levels of Nesting

| Level | What it is | Takes | Returns |
|-------|-----------|-------|--------|
| `retry(max_attempts=3)` | Factory | config args | decorator |
| `decorator(func)` | Decorator | function | wrapper |
| `wrapper(*args, **kwargs)` | Wrapper | call args | result |

## 1.8 Built-in: `@lru_cache` — Memoization

Python ships with a powerful caching decorator — like Spring's `@Cacheable`.
It caches results based on arguments, so repeated calls are **instant**.

In [27]:
# Without cache — exponential time O(2^n)
def fib_slow(n: int) -> int:
    if n < 2:
        return n
    return fib_slow(n - 1) + fib_slow(n - 2)
 
start = time.perf_counter()
result = fib_slow(30)
slow_time = time.perf_counter() - start
print(f"fib_slow(30) = {result}  ⏱️ {slow_time:.4f}s")
 
# With @lru_cache — O(n): each value computed once!
@lru_cache(maxsize=128)
def fib_fast(n: int) -> int:
    if n < 2:
        return n
    return fib_fast(n - 1) + fib_fast(n - 2)
 
start = time.perf_counter()
result = fib_fast(30)
fast_time = time.perf_counter() - start
print(f"fib_fast(30) = {result}  ⏱️ {fast_time:.6f}s")
print(f"\n🚀 Speedup: {slow_time / fast_time:.0f}x faster!")

fib_slow(30) = 832040  ⏱️ 0.2553s
fib_fast(30) = 832040  ⏱️ 0.000113s

🚀 Speedup: 2259x faster!


## 1.9 Exercises

**Exercise 1:** Write a `@count_calls` decorator that tracks how many times a function has been called.
Store the count as an attribute on the wrapper: `func.call_count`

In [27]:
# Your solution here
from functools import wraps

def count_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.call_count += 1
        print(f"The {func.__name__} called {wrapper.call_count} times")
        return func(*args, **kwargs)
        
    wrapper.call_count = 0 
    return wrapper

In [29]:
@count_calls
def add(a: float, b: float):
    return a + b

sum1 = add(5, 7)
print(sum1)
sum2 = add(3.2, 5.4)
print(sum2)
print(add.call_count)

The add called 1 times
12
The add called 2 times
8.600000000000001
2


**Exercise 2:** Write a `@slow_down(seconds)` **decorator factory** that adds a delay before each call.
Use it to throttle a function that prints numbers 1 to 5.

In [3]:
# Your solution here
import time
from functools import wraps 

def slow_down(seconds: int):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            time.sleep(seconds)
            return func(*args, **kwargs)
        return wrapper
    return decorator   

In [7]:
@slow_down(2)
def print_nums() -> None:
    for i in range(1, 6):
        print(i) 

print_nums()

1
2
3
4
5


## 📋 Takeaways

| # | Concept | Key Point |
|---|---------|----------|
| 1 | `@decorator` | Syntactic sugar for `func = decorator(func)` |
| 2 | Wrapper function | Takes `*args, **kwargs`, calls original, returns result |
| 3 | `@wraps(func)` | **Always** use it — preserves `__name__`, `__doc__` |
| 4 | `@timer` | `time.perf_counter()` before/after the call |
| 5 | `@debug` | Log calls with `repr()` of args and return |
| 6 | Stacking | `@A @B @C def f` → `A(B(C(f)))` — bottom-up |
| 7 | Decorator factory | Extra nesting layer when decorator needs config args |
| 8 | `@retry(n=3)` | Factory pattern: `retry()` → decorator → wrapper |
| 9 | `@lru_cache` | Built-in memoization — caches results by args |
| 10 | `.cache_info()` | See hits/misses; `.cache_clear()` to reset |
| 11 | Wrapper attributes | Store state on wrapper (e.g., `wrapper.call_count = 0`) |
| 12 | Java comparison | Decorators ≈ AOP + Proxy — but built into the language |